# TB Portals — Cohort Demographics Pull
Reads the TB Portals patient-level CSV to extract age, sex, and drug-resistance status per patient in our 5,010-image manifest, joined back to image-level rows. Returns a small CSV for local sub-population MAE analysis.

Attach: `tb-portals-cxr-pngs`. Runtime ≈ 1 min.

In [ ]:
import os, sys, subprocess
REPO_URL = 'https://github.com/mabdullahi7780/dl-project-codebase.git'
REPO_DIR = '/kaggle/working/dl-project-codebase'
BRANCH = 'cleaned-repo'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
for p in (REPO_DIR, REPO_DIR + '/scripts'):
    if p not in sys.path: sys.path.insert(0, p)
print('ready')

In [ ]:
import pandas as pd, os
from pathlib import Path
WORK = '/kaggle/working'
DATASET = '/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs'
KAGGLE_EXPORT = f'{DATASET}/kaggle_export'

# Find available CSVs in the dataset
import glob
all_csvs = sorted(glob.glob(f'{KAGGLE_EXPORT}/*.csv') + glob.glob(f'{DATASET}/**/*.csv', recursive=True))
for f in all_csvs:
    print(f)

In [ ]:
# Build manifest and try to join demographics if a patient-level CSV exists
import sys
if REPO_DIR + '/scripts' not in sys.path: sys.path.insert(0, REPO_DIR + '/scripts')
from build_paper_manifest import subsample
raw = pd.read_csv(f'{KAGGLE_EXPORT}/manifest.csv', dtype={'image_id': str, 'patient_id': str, 'country': str})
raw['image_path'] = raw['image_path'].apply(lambda p: p if str(p).startswith('/') else f'{KAGGLE_EXPORT}/{p}')
manifest = subsample(raw, seed=42)
print('manifest cols:', list(manifest.columns))
print('rows:', len(manifest))
manifest.head()

In [ ]:
# Try common TB Portals demographic CSV names
candidates = [
    f'{KAGGLE_EXPORT}/patient_demographics.csv',
    f'{KAGGLE_EXPORT}/TB_Portals_Patients_August_2023.csv',
    f'{KAGGLE_EXPORT}/patients.csv',
    f'{KAGGLE_EXPORT}/TB_Portals_CXR_Manual_Annotations_August_2023.csv',
]
demo = None
for c in candidates:
    if os.path.isfile(c):
        print('found:', c)
        demo = pd.read_csv(c, low_memory=False)
        print('cols:', list(demo.columns)[:30])
        break
if demo is None:
    print('No demographic CSV found; falling back to image-level info only.')

In [ ]:
# Build a per-image table with whatever demographic columns we can find
out_cols = ['image_id', 'patient_id', 'country', 'alp_0_100', 'cavity']
table = manifest[out_cols].copy()
if demo is not None:
    # try to find age, sex, MDR columns (case-insensitive substring match)
    lowmap = {c.lower(): c for c in demo.columns}
    def first_col(cands):
        for k in cands:
            for low, orig in lowmap.items():
                if k in low: return orig
        return None
    age_col  = first_col(['age'])
    sex_col  = first_col(['sex', 'gender'])
    mdr_col  = first_col(['drug_resistance', 'mdr', 'resistance'])
    pid_col  = first_col(['patient_id', 'condition_id'])
    print('age:', age_col, '| sex:', sex_col, '| mdr:', mdr_col, '| pid:', pid_col)
    if pid_col is not None:
        keep = [pid_col] + [c for c in [age_col, sex_col, mdr_col] if c is not None]
        demo_small = demo[keep].drop_duplicates(subset=[pid_col])
        demo_small = demo_small.rename(columns={pid_col: 'patient_id',
                                                age_col or 'age': 'age',
                                                sex_col or 'sex': 'sex',
                                                mdr_col or 'mdr': 'mdr'})
        demo_small['patient_id'] = demo_small['patient_id'].astype(str)
        table['patient_id'] = table['patient_id'].astype(str)
        table = table.merge(demo_small, on='patient_id', how='left')
        print('joined; non-null age:', table.get('age', pd.Series()).notna().sum(),
              'sex:', table.get('sex', pd.Series()).notna().sum(),
              'mdr:', table.get('mdr', pd.Series()).notna().sum())
table.to_csv(f'{WORK}/cohort_demographics.csv', index=False)
table.head()

In [ ]:
import shutil
shutil.copy(f'{WORK}/cohort_demographics.csv', f'{WORK}/cohort_demographics_dl.csv')
print('saved to', f'{WORK}/cohort_demographics.csv')